# Getting the Retrieval Corpus

## 1. Importing Python Document Downloading Libraries

In [1]:
#!/usr/bin/env python3
"""
Kenyan civic-information RAG: knowledge-base downloader (max 5 documents per source).

A "source" is an issuing body (Kenya Law, IEBC, National Treasury, Controller of
Budget, KNBS, Government Printer, ...). No more than MAX_PER_SOURCE documents are
kept for each, counting files already recorded in the manifest, so re-runs never
exceed the cap.

Two acquisition modes per entry:
  1. direct - a known document URL.
  2. crawl  - a landing page; collects PDF links whose text/URL match keywords.

Every file gets a provenance record (JSONL manifest) so answers can be cited later:
issuing body, doc type, URL, retrieval date, SHA-256.

Usage:
    pip install requests beautifulsoup4 tqdm
    python kenya_kb_downloader.py --out ./kb_raw                 # download
    python kenya_kb_downloader.py --out ./kb_raw --check         # test URLs only
    python kenya_kb_downloader.py --out ./kb_raw --max-per-source 3

URLs on Kenyan portals move often: treat SOURCES as seeds, run --check, fix dead
links. Respect each site's robots.txt and terms of use.
"""
import argparse
import hashlib
import json
import re
import time
import urllib.robotparser as rp
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm


## 2. Defining Document Sources using Dictionaries

In [2]:
UA = "EduResearchBot/0.1 (academic RAG project; contact: benir.omenda@strathmore.edu)"
DELAY_S = 2.0            # politeness delay between requests to the same host
TIMEOUT = 60
MAX_MB = 150
MAX_PER_SOURCE = 5       # hard cap per issuing body

# ---------------------------------------------------------------------------
# SOURCE REGISTRY: category -> entries. The cap applies per "body", across entries.
# direct: {"mode","name","url","body","doc_type"}
# crawl : {"mode","name","page","body","doc_type","include":[regex],"keywords":[str],"max":int}
# ---------------------------------------------------------------------------
SOURCES = {
    "legislation": [  # Kenya Law: Constitution + 4 Acts = exactly 5
        {"mode": "direct", "name": "Constitution of Kenya 2010",
         "url": "http://kenyalaw.org/kl/fileadmin/pdfdownloads/TheConstitutionOfKenya.pdf",
         "body": "Kenya Law", "doc_type": "constitution"},
        {"mode": "crawl", "name": "Elections Act (No. 24 of 2011)",
         "page": "https://new.kenyalaw.org/akn/ke/act/2011/24/eng",
         "body": "Kenya Law", "doc_type": "act",
         "include": [r"\.pdf$", r"/source$", r"/media/"], "max": 1},
        {"mode": "crawl", "name": "Election Offences Act (No. 37 of 2016)",
         "page": "https://new.kenyalaw.org/akn/ke/act/2016/37/eng",
         "body": "Kenya Law", "doc_type": "act",
         "include": [r"\.pdf$", r"/source$", r"/media/"], "max": 1},
        {"mode": "crawl", "name": "Leadership and Integrity Act (No. 19 of 2012)",
         "page": "https://new.kenyalaw.org/akn/ke/act/2012/19/eng",
         "body": "Kenya Law", "doc_type": "act",
         "include": [r"\.pdf$", r"/source$", r"/media/"], "max": 1},
        {"mode": "crawl", "name": "Political Parties Act (No. 11 of 2011)",
         "page": "https://new.kenyalaw.org/akn/ke/act/2011/11/eng",
         "body": "Kenya Law", "doc_type": "act",
         "include": [r"\.pdf$", r"/source$", r"/media/"], "max": 1},
    ],
    "iebc": [
        {"mode": "crawl", "name": "IEBC publications",
         "page": "https://www.iebc.or.ke/resources/",
         "body": "IEBC", "doc_type": "iebc_publication",
         "include": [r"\.pdf$"],
         "keywords": ["voter education", "operations plan", "registration", "tallying",
                      "results", "regulations", "code of conduct", "timetable"],
         "max": 5},
    ],
    "gazette": [
        {"mode": "crawl", "name": "Kenya Gazette (Special Issues, election-related)",
         "page": "http://kenyalaw.org/kenya_gazette/",
         "body": "Government Printer (Kenya Gazette)", "doc_type": "gazette_special",
         "include": [r"\.pdf$"],
         "keywords": ["special issue", "election", "iebc", "candidate", "nomination",
                      "spending limit", "campaign"],
         "max": 5},
    ],
    "manifestos": [
        # No single official page. Add verified direct links, e.g.:
        # {"mode":"direct","name":"Kenya Kwanza Manifesto 2022","url":"<verified>",
        #  "body":"UDA","doc_type":"manifesto"},
        # {"mode":"direct","name":"Azimio Manifesto 2022","url":"<verified>",
        #  "body":"Azimio","doc_type":"manifesto"},
    ],
    "public_finance": [
        {"mode": "crawl", "name": "National Treasury budget documents",
         "page": "https://www.treasury.go.ke/budget-documents/",
         "body": "National Treasury", "doc_type": "budget",
         "include": [r"\.pdf$"],
         "keywords": ["budget review and outlook", "appropriation", "outturn",
                      "budget highlights", "citizen"],
         "max": 5},
        {"mode": "crawl", "name": "Controller of Budget county reports",
         "page": "https://cob.go.ke/reports/consolidated-county-budget-implementation-review-reports/",
         "body": "Controller of Budget", "doc_type": "cob_report",
         "include": [r"\.pdf$"], "max": 5},
        {"mode": "crawl", "name": "KNBS Economic Survey",
         "page": "https://www.knbs.or.ke/",
         "body": "KNBS", "doc_type": "economic_survey",
         "include": [r"\.pdf$"],
         "keywords": ["economic survey", "statistical abstract", "gross county product",
                      "poverty"],
         "max": 5},
    ],
}

In [3]:
session = requests.Session()
session.headers.update({"User-Agent": UA})
_robots = {}
_last_hit = {}


def allowed(url: str) -> bool:
    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    if base not in _robots:
        r = rp.RobotFileParser()
        try:
            r.set_url(base + "/robots.txt")
            r.read()
        except Exception:
            r = None
        _robots[base] = r
    r = _robots[base]
    return True if r is None else r.can_fetch(UA, url)


def polite_get(url, stream=False, retries=3):
    host = urlparse(url).netloc
    if not allowed(url):
        raise PermissionError(f"robots.txt disallows {url}")
    for attempt in range(1, retries + 1):
        wait = DELAY_S - (time.time() - _last_hit.get(host, 0))
        if wait > 0:
            time.sleep(wait)
        try:
            resp = session.get(url, timeout=TIMEOUT, stream=stream, allow_redirects=True)
            _last_hit[host] = time.time()
            resp.raise_for_status()
            return resp
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)


## 3. Finding Links

In [4]:
def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")[:90]


def find_links(entry, limit):
    """Crawl a landing page; return up to `limit` [(link_text, absolute_url)]."""
    html = polite_get(entry["page"]).text
    soup = BeautifulSoup(html, "html.parser")
    inc = [re.compile(p, re.I) for p in entry.get("include", [r"\.pdf$"])]
    kws = [k.lower() for k in entry.get("keywords", [])]
    out, seen = [], set()
    for a in soup.find_all("a", href=True):
        url = urljoin(entry["page"], a["href"])
        text = " ".join(a.get_text(" ", strip=True).split())
        if url in seen or not any(p.search(url) for p in inc):
            continue
        hay = (text + " " + url).lower()
        if kws and not any(k in hay for k in kws):
            continue
        seen.add(url)
        # for single-document crawls, name the file after the entry, not the link text
        out.append((text or Path(urlparse(url).path).name, url))
        if len(out) >= limit:
            break
    return out


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

## Downloading the files

In [5]:
def download(url, dest: Path):
    with polite_get(url, stream=True) as r:
        size = int(r.headers.get("Content-Length", 0))
        if size > MAX_MB * 1024 * 1024:
            raise ValueError(f"file too large ({size / 1e6:.0f} MB)")
        ctype = r.headers.get("Content-Type", "")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1 << 16):
                f.write(chunk)
    return ctype


def load_manifest(path: Path):
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


def run(out_dir: Path, categories, check_only: bool, cap: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.jsonl"
    existing = load_manifest(manifest_path)
    known_urls = {r["url"] for r in existing}
    per_body = Counter(r["issuing_body"] for r in existing)   # persists across runs

    report = {"ok": 0, "skipped": 0, "capped": 0, "failed": []}
    with open(manifest_path, "a", encoding="utf-8") as mf:
        for cat in categories:
            for entry in SOURCES.get(cat, []):
                body = entry["body"]
                room = cap - per_body[body]
                if room <= 0:
                    print(f"[CAP] {body}: already {cap} documents, skipping {entry['name']}")
                    report["capped"] += 1
                    continue
                try:
                    if entry["mode"] == "direct":
                        targets = [(entry["name"], entry["url"])]
                    else:
                        targets = find_links(entry, min(entry.get("max", cap), room))
                except Exception as e:
                    report["failed"].append((entry["name"], str(e)))
                    print(f"[FAIL] {entry['name']}: {e}")
                    continue
                if not targets:
                    print(f"[WARN] no links found for {entry['name']}: check selectors/page")

                for title, url in tqdm(targets, desc=entry["name"][:40]):
                    if per_body[body] >= cap:
                        report["capped"] += 1
                        break
                    if url in known_urls:
                        report["skipped"] += 1
                        continue
                    if check_only:
                        try:
                            polite_get(url, stream=True).close()
                            report["ok"] += 1
                            per_body[body] += 1
                        except Exception as e:
                            report["failed"].append((url, str(e)))
                        continue
                    # use the entry name for single-doc crawls so files are readable
                    label = entry["name"] if entry.get("max") == 1 else title
                    ext = Path(urlparse(url).path).suffix or ".pdf"
                    dest = out_dir / cat / f"{slugify(label) or 'doc'}{ext}"
                    try:
                        ctype = download(url, dest)
                        digest = sha256_file(dest)
                        rec = {
                            "doc_id": digest[:16],
                            "title": label,
                            "category": cat,
                            "doc_type": entry["doc_type"],
                            "issuing_body": body,
                            "url": url,
                            "landing_page": entry.get("page"),
                            "local_path": str(dest),
                            "content_type": ctype,
                            "sha256": digest,
                            "retrieved_at": datetime.now(timezone.utc).isoformat(),
                        }
                        mf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        mf.flush()
                        known_urls.add(url)
                        per_body[body] += 1
                        report["ok"] += 1
                    except Exception as e:
                        report["failed"].append((url, str(e)))
                        print(f"[FAIL] {url}: {e}")

    print(f"\nDone. ok={report['ok']} skipped={report['skipped']} "
          f"capped={report['capped']} failed={len(report['failed'])}")
    print("Documents per source:", dict(per_body))
    for u, err in report["failed"]:
        print(f"  - {u}: {err}")

In [7]:
ap = argparse.ArgumentParser()
    # Default: ../resources, i.e. a sibling of the notebooks/ folder you run from.
ap.add_argument("--out", default="../resources")
ap.add_argument("--only", nargs="*", choices=list(SOURCES), help="limit categories")
ap.add_argument("--check", action="store_true", help="test links without saving")
ap.add_argument("--max-per-source", type=int, default=MAX_PER_SOURCE)
a, _unknown = ap.parse_known_args()
out = Path(a.out).resolve()
print(f"Saving to: {out}")
run(out, a.only or list(SOURCES), a.check, a.max_per_source)

Saving to: /workspace/ICSAI/SemProject_MLEngine/resources


Constitution of Kenya 2010: 100%|██████████| 1/1 [00:08<00:00,  8.19s/it]

[FAIL] http://kenyalaw.org/kl/fileadmin/pdfdownloads/TheConstitutionOfKenya.pdf: 403 Client Error: Forbidden for url: https://kenyalaw.org/kl/fileadmin/pdfdownloads/TheConstitutionOfKenya.pdf


[WARN] no links found for Elections Act (No. 24 of 2011): check selectors/page


Elections Act (No. 24 of 2011): 0it [00:00, ?it/s]


...

[WARN] no links found for IEBC publications: check selectors/page


IEBC publications: 0it [00:00, ?it/s]


[FAIL] Kenya Gazette (Special Issues, election-related): 403 Client Error: Forbidden for url: https://kenyalaw.org/kenya_gazette/
[FAIL] National Treasury budget documents: 404 Client Error: Not Found for url: https://www.treasury.go.ke/budget-documents/
[WARN] no links found for Controller of Budget county reports: check selectors/page


Controller of Budget county reports: 0it [00:00, ?it/s]


[FAIL] KNBS Economic Survey: HTTPSConnectionPool(host='www.knbs.or.ke', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))

Done. ok=0 skipped=0 capped=0 failed=4
Documents per source: {}
  - http://kenyalaw.org/kl/fileadmin/pdfdownloads/TheConstitutionOfKenya.pdf: 403 Client Error: Forbidden for url: https://kenyalaw.org/kl/fileadmin/pdfdownloads/TheConstitutionOfKenya.pdf
  - Kenya Gazette (Special Issues, election-related): 403 Client Error: Forbidden for url: https://kenyalaw.org/kenya_gazette/
  - National Treasury budget documents: 404 Client Error: Not Found for url: https://www.treasury.go.ke/budget-documents/
  - KNBS Economic Survey: HTTPSConnectionPool(host='www.knbs.or.ke', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify f